In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))
from src.adapter.api.template.comment import SpecialAbilitiesComment

In [2]:
from src.adapter.database.postgres_repository import PostgresStudentRepository
from src.adapter.database.postges_manager import postgres_manager
student_repo = PostgresStudentRepository(postgres_manager.session)
(student_repo.exists({"name": 'Võ Minh Khang'})).code

2026-05-15 16:43:11,804 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-05-15 16:43:11,804 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-05-15 16:43:12,058 INFO sqlalchemy.engine.Engine select current_schema()
2026-05-15 16:43:12,058 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-05-15 16:43:12,323 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-05-15 16:43:12,327 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-05-15 16:43:12,569 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-05-15 16:43:12,585 INFO sqlalchemy.engine.Engine SELECT student.code AS student_code, student.name AS student_name, student.card_id AS student_card_id, student.edu_id AS student_edu_id, student.date_of_birth AS student_date_of_birth, student.gender AS student_gender, student.ethnicity AS student_ethnicity, student.status AS student_status, student.phone AS student_phone, student.nationality AS student_nationality, student.address AS student_address, student.place_of_birth

'20252026.01.Mo1.011'

In [3]:
from src.adapter.database.postges_manager import postgres_manager
from src.adapter.database.mongo_manager import mongo_manager
from src.adapter.graph.neo4j_manager import neo4j_manager
from src.application.core import SystemCore

session = postgres_manager.session
db = mongo_manager.get_metadata_db()
manager = neo4j_manager
repo = SystemCore(session, db, manager)

d:\Work\Partner\hoc-ba-so\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3349.13it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
import re
from openpyxl import load_workbook
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Optional
import json
import re

from src.common.postgres_model_setting import settings

def normalize_text(value: object) -> str:
    if value is None:
        return ""
    text = str(value).replace("\n", " ").strip()
    text = re.sub(r"\s+", " ", text)
    return text


def extract_student_name(raw_text: str) -> str:
    text = normalize_text(raw_text)
    if ":" in text:
        return text.split(":", 1)[1].strip()
    return text

def extract_class_room(raw_text: str) -> str:
    text = normalize_text(raw_text)
    if ":" in text:
        text = text.split(":", 1)[1].strip()
    text = text.split(" ")
    return f"{settings.WORD2SHORTCUT[text[0]]}{text[1]}"

def find_header_row_and_comment_col(ws) -> tuple[int, int]:
    """
    Tìm dòng header có:
    - cột tên môn học
    - cột nhận xét
    """
    for row_idx in range(1, ws.max_row + 1):
        row_values = [normalize_text(ws.cell(row_idx, col_idx).value) for col_idx in range(1, ws.max_column + 1)]

        subject_col = None
        comment_col = None

        for col_idx, value in enumerate(row_values, start=1):
            value_lower = value.lower()
            if "môn học" in value_lower and "hoạt động giáo dục" in value_lower:
                subject_col = col_idx
            if "nhận xét" in value_lower:
                comment_col = col_idx

        if subject_col and comment_col:
            return row_idx, comment_col

    raise ValueError("Không tìm thấy dòng header hoặc cột 'Nhận xét' trong sheet.")


def extract_comments_from_sheet(
    file_path: str | Path,
    sheet_name: str,
    subject_name: str
):
    wb = load_workbook(file_path, data_only=True)
    if sheet_name not in wb.sheetnames:
        raise ValueError(f"Không tìm thấy sheet: {sheet_name}")

    ws = wb[sheet_name]

    # Tên học sinh thường ở A1: "Họ và tên học sinh: Trương Phúc An"
    student_name = extract_student_name(ws["A1"].value)
    class_room = extract_class_room(ws["O1"].value)
    # print(class_room)

    header_row, comment_col = find_header_row_and_comment_col(ws)
    subject_col = 1  # trong file này môn học nằm ở cột A

    code = student_repo.exists({"name": student_name,
                                "code": f"LIKE %{class_room}%"}).code
    print(code)
    # subject_to_field = {
    #     "tiếng việt",
    #     "toán",
    #     "th-cn (tin học)",
    #     "tin học",
    #     "khoa học",
    #     "lịch sử và địa lí",
    #     "ngoại ngữ tiếng anh",
    #     "tiếng anh",
    #     "công nghệ",
    #     "nghệ thuật (âm nhạc)",
    #     "âm nhạc",
    #     "nghệ thuật (mĩ thuật, mỹ thuật)",
    #     "mĩ thuật",
    #     "mỹ thuật",
    #     "đạo đức",
    #     "giáo dục thể chất",
    #     "hoạt động trải nghiệm",
    #     "tự nhiên và xã hội"
    # }
    for row_idx in range(header_row + 1, ws.max_row + 1):
        subject = normalize_text(ws.cell(row_idx, subject_col).value)
        comment = normalize_text(ws.cell(row_idx, comment_col).value)

        if not subject:
            continue

        subject_key = subject.lower()
        if subject_name in subject_key:
            return {
                "student_code": code,
                "comment": comment
            }
    return {
        "student_code": code,
        "comment": ""
    }

# if __name__ == "__main__":
    # for file_path in excel_files:

In [5]:
from pathlib import Path

from pathlib import Path

folder_path = Path("../raw_data/Ho_ba_hoc_sinh/")

excel_files = [str(f) for f in folder_path.rglob("*.xlsx")]

SUBJECT_NAME = "nghệ thuật (mĩ thuật)"
print(excel_files)
data_save = []
for file_path in excel_files:
    signal_number = file_path.split("\\")[-2][-1]
    signal_class = file_path.split("\\")[-2][:-1]
    signal = f"{signal_class} {signal_number}"
    print(signal)
    data = extract_comments_from_sheet(file_path, sheet_name=f"MH {signal} (2024-2025)", subject_name=SUBJECT_NAME)
    data_save.append(data)
    # break
# with open(f'../json/20252026.03.Ba1.020.json', "w", encoding="utf-8") as f:
#     json.dump(data_json, f, ensure_ascii=False, indent=4)
with open(f'../json_raw/{SUBJECT_NAME}.json', "w", encoding="utf-8") as f:
    json.dump(data_save, f, ensure_ascii=False, indent=4)

['..\\raw_data\\Ho_ba_hoc_sinh\\Ba1\\HOC_BA_10_NGUYENTHAIBAOHY_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\Ba1\\HOC_BA_11_NGUYENHOANGXUANKHANH_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\Ba1\\HOC_BA_12_HOANGTHIENKHIEM_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\Ba1\\HOC_BA_13_LEGIAKIET_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\Ba1\\HOC_BA_14_NGOGIALAC_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\Ba1\\HOC_BA_15_CHAUCHILAM_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\Ba1\\HOC_BA_16_LENGOCLINH_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\Ba1\\HOC_BA_17_TRINHHALINH_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\Ba1\\HOC_BA_18_TRUONGGIALINH_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\Ba1\\HOC_BA_19_NGUYENBAONAM_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\Ba1\\HOC_BA_1_PHAMQUYNHANH_NAMHOC_20242025.xlsx', '..\\raw_data\\Ho_ba_hoc_sinh\\Ba1\\HOC_BA_20_LEPHUCNGUYEN_NAMHOC_20242025.xlsx', '..\\raw_dat